In [ ]:
# ============================================================
# STEP 1 — imports + connect to database
# ============================================================
import pandas as pd
import geopandas as gpd
import sqlite3
import numpy as np




db_Path = r"C:\cabarrus-gis-prep\sql\databases\cabarrus_recreation.db"
conn = sqlite3.connect(db_Path)

#HERE IM SKIPPING THE creation of databases because the park data is already a table in the db_path. But usually you would need to 
#pd.read_csv() the data and then use to_sql() to put it in the database. Check park_analysisPractice that where I did that.



print(pd.read_sql_query("SELECT * FROM parks LIMIT 5",conn))

              X              Y  OBJECTID FACILITYID  \
0  1.515174e+06  644107.563114         1      FAC-1   
1  1.517086e+06  590589.513982         2      FAC-6   
2  1.572810e+06  603840.000112         3     FAC-19   
3  1.525360e+06  601560.000571         4     FAC-12   
4  1.556115e+06  626720.964439         5      FAC-4   

                                              NAME  SUBTYPEFIELD  \
0  Bakers Creek Park/Greenway - City of Kannapolis           820   
1               Frank Liske Park - Cabarrus County           820   
2                                  McAllister Park           820   
3   Hartsell Park and Rec Center - City of Concord           820   
4                   Camp Spencer - Cabarrus County           820   

      FEATURECODE                             FULLADDR   OPERDAYS  \
0  Municipal Park         1275 West A St, 704-920-4343  Sun - Sat   
1     County Park         4001 Stough Rd, 704-920-3350  Sun - Sat   
2            Park                         8595 Park D

In [ ]:
# ============================================================
# STEP 2 — SQL QUERY
# Pull only the columns we need from the database
# WHERE X IS NOT NULL AND Y IS NOT NULL makes sure
# every park has coordinates — no coordinates = can't map it
# ============================================================
query = """
SELECT NAME, FULLADDR, FEATURECODE,
       PLAYGROUND, BASEBALL, BASKETBALL,
       SOCCER, HIKING, PICNIC, FISHING,
       BOATING, RESTROOM, ADACOMPLY,
       SWIMMING, CAMPING, GOLF,
       PARKAREA, X, Y
FROM parks
WHERE X IS NOT NULL
AND Y IS NOT NULL
"""

# pd.read_sql_query runs the SQL and stores results in df
# df is now a Pandas DataFrame — a table in memory
df = pd.read_sql_query(query, conn)

print(f"Parks loaded: {len(df)}")

print("\nPREVIEW OF FIRST 3 PARKS")
print(df.head(3).T)  


Parks loaded: 57

PREVIEW OF FIRST 3 PARKS
                                                           0  \
NAME         Bakers Creek Park/Greenway - City of Kannapolis   
FULLADDR                        1275 West A St, 704-920-4343   
FEATURECODE                                   Municipal Park   
PLAYGROUND                                               Yes   
BASEBALL                                                 Yes   
BASKETBALL                                               Yes   
SOCCER                                                    No   
HIKING                                                   Yes   
PICNIC                                                   Yes   
FISHING                                                   No   
BOATING                                                   No   
RESTROOM                                                 Yes   
ADACOMPLY                                                Yes   
SWIMMING                                                  No 

In [ ]:
# ============================================================
# STEP 3 — CREATE NEW COLUMNS
# SQL only gave us the raw Yes/No data
# Now we calculate new columns from that data
# You MUST run this before trying to print AMENITY_SCORE
# because the column doesn't exist until this code runs
# ============================================================

# List of amenity columns to count
amenity_cols = ['PLAYGROUND','BASEBALL','BASKETBALL',
                'SOCCER','HIKING','PICNIC',
                'FISHING','BOATING','SWIMMING',
                'CAMPING','GOLF']

# How this line works:
# df[col] == 'Yes'   → True or False for each row
# .astype(int)       → converts True→1, False→0
# sum(...)           → adds up all the 1s per row
# Result: each park gets a number 0-11
# 0 = no amenities, 11 = has everything
df['AMENITY_SCORE'] = sum(
    (df[col] == 'Yes').astype(int) for col in amenity_cols
)

# ============================================================
# STEP 3 — DISPLAY THE NEW COLUMN
# Now AMENITY_SCORE exists so we can print it
# double bracket because we want table format
# ============================================================
print("\nPARK AMENITY SCORES")
print(df[['NAME', 'AMENITY_SCORE']].to_string())

# Quick math check using single bracket
# single bracket because we want a clean plain number
avg = df['AMENITY_SCORE'].mean()
highest = df['AMENITY_SCORE'].max()
lowest = df['AMENITY_SCORE'].min()

print(f"\nAverage amenity score: {avg:.2f}")
print(f"Highest scoring park: {highest}")
print(f"Lowest scoring park: {lowest}")


# ============================================================
# BRACKET RULES — read this whenever you forget
#
# SINGLE BRACKET df['col']
# → returns a Series (like a plain list, no header)
# → use for math: df['AMENITY_SCORE'].mean()
# → result is a clean plain number — easy to use elsewhere
# → example output:
#      0    3
#      1    6
#      Name: AMENITY_SCORE, dtype: int64
#
# DOUBLE BRACKET df[['col']]
# → returns a DataFrame (table format with header on top)
# → use for displaying: print(df[['NAME', 'AMENITY_SCORE']])
# → result has column name as header — looks like a spreadsheet
# → example output:
#      AMENITY_SCORE
#   0            3
#   1            6
#
# WHY SINGLE IS BETTER FOR MATH:
# df['AMENITY_SCORE'].mean() → returns 2.96 (clean number)
# df[['AMENITY_SCORE']].mean() → returns:
#      AMENITY_SCORE    2.96
#      dtype: float64
# That extra wrapping causes errors if you try to use
# the result in calculations or f-strings
#
# SIMPLE RULE:
# Math or calculations  → single bracket  df['col']
# Displaying as table   → double bracket  df[['col1','col2']]
# ============================================================


                                               NAME     FEATURECODE
0   Bakers Creek Park/Greenway - City of Kannapolis  Municipal Park
1                Frank Liske Park - Cabarrus County     County Park
2                                   McAllister Park            Park
3    Hartsell Park and Rec Center - City of Concord  Municipal Park
4                    Camp Spencer - Cabarrus County     County Park
5                Veterans Park - City of Kannapolis  Municipal Park
6              Beverly Hills Park - City of Concord  Municipal Park
7                 Village Park - City of Kannapolis  Municipal Park
8                 W.W. Flowe Park - City of Concord  Municipal Park
9                       Beach Springs Mt. Bike Park            Park
10        Pharr Mill Road Park - Town of Harrisburg  Municipal Park
11        Harrisburg Town Park - Town of Harrisburg  Municipal Park
12         Stallings Road Park - Town of Harrisburg  Municipal Park
13                       AT Allen Elementary Sch

In [ ]:
# ============================================================
# STEP 4 — ADD SIZE_CAT AND ADA_LABEL COLUMNS
# np.select = same as CASE WHEN in SQL
# conditions = list of if statements checked top to bottom
# choices = what to assign when condition is true
# default = what to use when none of the conditions match
# (covers parks where PARKAREA is null = Unknown)
# ============================================================

conditions = [
    df['PARKAREA'] < 10,
    (df['PARKAREA'] >= 10) & (df['PARKAREA'] < 50),
    (df['PARKAREA'] >= 50) & (df['PARKAREA'] < 200),
    df['PARKAREA'] >= 200
]
choices = ['Small', 'Medium', 'Large', 'Regional']

df['SIZE_CAT'] = np.select(conditions, choices, default='Unknown')

# np.where = same as a simple IF/ELSE
# if ADACOMPLY = Yes → 'ADA Compliant'
# anything else      → 'Not ADA Compliant'
df['ADA_LABEL'] = np.where(
    df['ADACOMPLY'] == 'Yes',
    'ADA Compliant',
    'Not ADA Compliant'
)

# Verify new columns exist and look right
print("NEW COLUMNS ADDED:")
print(df[['NAME', 'AMENITY_SCORE', 'SIZE_CAT', 'ADA_LABEL']].to_string())